# Project Bài 3 · Chọn ngưỡng cho bộ lọc tin rác theo mục tiêu vận hành

**Học máy cơ bản · Bài 3 · PGS.TS Nguyễn Văn Hậu**

**Tình huống.** Bộ lọc kNN của Bài 2 cho mỗi tin một số phiếu spam v (0–11). Nhóm vận hành phải chọn ngưỡng m (gắn cờ spam khi v ≥ m) theo một mục tiêu đã thống nhất: *tránh chặn nhầm* hay *tránh bỏ lọt*. Bạn tự viết các hàm đánh giá, chọn ngưỡng đúng quy trình và viết báo cáo một trang có giới hạn.

Project **độc lập** với project Bài 2: số phiếu đã được cấp sẵn trong `data/sms_validation_votes_v3.csv` và `data/sms_test_votes_v3.csv` (kNN tất định của Bài 2 v3, k = 11).

| Mốc | Bước | Nộp | Ô tự kiểm |
|---|---|---|---|
| M1 | 1–3: dữ liệu, ma trận, chỉ số | ngay sau buổi học | số tin, số spam; 10 tin mô phỏng; validation m = 6 |
| M2 | 4–5: quét ngưỡng, chọn theo chi phí | tuần sau | bảng 12 ngưỡng; quy luật đơn điệu; ngưỡng được chọn |
| M3 | 6: chấm test một lần, báo cáo | cuối đợt | `cham_test` được gọi đúng một lần |

**Ba mức.** *Cơ bản*: kịch bản 5·FP + FN; đặt `LAM_MUC_KHA = False` ở bước 5. *Khá*: thêm kịch bản FP + 5·FN (khoá cả hai ngưỡng trên validation rồi chấm test một lần cho cả hai) và phần đối chiếu scikit-learn. *Nâng cao*: phần cuối notebook (AUC so cặp, AP, tỷ lệ lớp).

**Cách dùng.** Chạy từ trên xuống. Ô có `TODO` là phần bạn viết. Ô bắt đầu bằng `# TỰ KIỂM` không được sửa: nó in ✅ khi đúng, hoặc báo lỗi kèm gợi ý. Ô tự kiểm chỉ kiểm phép tính; phần diễn giải được chấm theo rubric.

**Quy ước.** Spam là lớp dương (1). Ma trận theo thứ tự nhãn [0, 1]: hàng là nhãn thật, cột là dự đoán. **Chọn ngưỡng chỉ trên validation**, theo tiêu chí và quy tắc hoà ghi trước. **Test chỉ chấm một lần.**

In [ ]:
# Thiết lập: chạy ô này trước
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
assert (ROOT / 'data' / 'sms_validation_votes_v3.csv').exists(), 'Mở notebook từ thư mục bài (có thư mục data/).'

def kiem(ten, dieu_kien, goi_y=''):
    # Ô tự kiểm: in ✅ khi đúng; báo lỗi kèm gợi ý khi sai.
    if not dieu_kien:
        raise AssertionError(f'❌ {ten}. Gợi ý: {goi_y}')
    print(f'✅ {ten}')

## Bước 1 · Đọc dữ liệu, ghép văn bản, mốc “luôn ham” (Chặng 1)

Mỗi tệp số phiếu có các cột `source_record` (số dòng trong tệp gốc, đếm từ 1), `y` (1 = spam), `votes` (số phiếu spam trong 11 láng giềng). Văn bản lấy từ `data/SMSSpamCollection` theo `source_record`.

In [ ]:
val = pd.read_csv(ROOT / 'data' / 'sms_validation_votes_v3.csv')
test = pd.read_csv(ROOT / 'data' / 'sms_test_votes_v3.csv')
goc = pd.read_csv(ROOT / 'data' / 'SMSSpamCollection', sep='\t', names=['label', 'text'], quoting=3)
goc['source_record'] = np.arange(1, len(goc) + 1)
val = val.merge(goc[['source_record', 'text']], on='source_record', how='left')
test = test.merge(goc[['source_record', 'text']], on='source_record', how='left')
# TODO: accuracy của quy tắc "luôn trả lời ham" trên validation
acc_luon_ham_val = ...
raise NotImplementedError('TODO bước 1: tính acc_luon_ham_val rồi xoá dòng raise này')
print('validation:', len(val), 'tin,', val.y.sum(), 'spam · test:', len(test), 'tin,', test.y.sum(), 'spam')
print('Mốc luôn ham trên validation:', round(acc_luon_ham_val, 4))

In [ ]:
# TỰ KIỂM · bước 1
kiem('Validation có 1032 tin, 129 spam', len(val) == 1032 and int(val.y.sum()) == 129, 'Đọc đúng tệp sms_validation_votes_v3.csv.')
kiem('Test có 1032 tin, 128 spam', len(test) == 1032 and int(test.y.sum()) == 128, 'Đọc đúng tệp sms_test_votes_v3.csv.')
kiem('Số phiếu nằm trong 0…11', val.votes.between(0, 11).all() and test.votes.between(0, 11).all(), '')
kiem('Mọi tin đều ghép được văn bản', val.text.notna().all() and test.text.notna().all(), 'Ghép theo source_record; tệp gốc đọc với sep="\\t", quoting=3.')
kiem('Mốc luôn ham trên validation ≈ 0.8750', abs(acc_luon_ham_val - 0.875) < 1e-9,
     'Accuracy của "luôn ham" = số tin ham / số tin.')

## Bước 2 · Tự viết ma trận nhầm lẫn (Chặng 1)

Viết `ma_tran(y, y_du_doan)` trả về bốn số `(tn, fp, fn, tp)`, **không dùng scikit-learn**. Ví dụ 10 tin mô phỏng của slide: số phiếu A–J là 11, 10, 9, 8, 7, 6, 4, 3, 2, 1; spam ở A, C, D, F.

In [ ]:
def ma_tran(y, y_du_doan):
    y = np.asarray(y).astype(int); y_du_doan = np.asarray(y_du_doan).astype(int)
    # TODO: đếm bốn ô và trả về (tn, fp, fn, tp)
    raise NotImplementedError('TODO bước 2: viết ma_tran')

toy_v = np.array([11, 10, 9, 8, 7, 6, 4, 3, 2, 1]); toy_y = np.array([1, 0, 1, 1, 0, 1, 0, 0, 0, 0])

In [ ]:
# TỰ KIỂM · bước 2
kiem('10 tin, m = 6: (4, 2, 0, 4)', ma_tran(toy_y, toy_v >= 6) == (4, 2, 0, 4), 'TN: thật 0, đoán 0; FP: thật 0, đoán 1; FN: thật 1, đoán 0; TP: thật 1, đoán 1.')
kiem('10 tin, m = 9: (5, 1, 2, 2)', ma_tran(toy_y, toy_v >= 9) == (5, 1, 2, 2), 'Ở m = 9 chỉ A, B, C bị gắn cờ.')
kiem('Validation, m = 6: (894, 9, 16, 113)', ma_tran(val.y, val.votes >= 6) == (894, 9, 16, 113), 'Dự đoán spam khi votes ≥ 6.')
kiem('Tổng bốn ô bằng số tin', sum(ma_tran(val.y, val.votes >= 6)) == len(val), '')

## Bước 3 · Tự viết các chỉ số (Chặng 2)

Viết `chi_so(tn, fp, fn, tp)` trả về dict với `accuracy`, `precision`, `recall`, `f1`. Khi mẫu số bằng 0 (không có cảnh báo, hoặc không có spam thật), trả `None` và **ghi rõ** quy ước này trong báo cáo; đừng âm thầm thay bằng 0.

In [ ]:
def chi_so(tn, fp, fn, tp):
    # TODO: accuracy, precision, recall, f1; None khi mẫu số bằng 0
    raise NotImplementedError('TODO bước 3: viết chi_so')

In [ ]:
# TỰ KIỂM · bước 3
c = chi_so(*ma_tran(toy_y, toy_v >= 6))
kiem('10 tin, m = 6: P = 2/3, R = 1, F1 = 0,8', abs(c['precision'] - 2 / 3) < 1e-12 and c['recall'] == 1 and abs(c['f1'] - .8) < 1e-12, 'P = TP/(TP+FP); R = TP/(TP+FN); F1 = 2TP/(2TP+FP+FN).')
c = chi_so(*ma_tran(val.y, val.votes >= 6))
kiem('Validation, m = 6: P ≈ 0.9262, R ≈ 0.8760', abs(c['precision'] - 0.9262295081967213) < 1e-9 and abs(c['recall'] - 0.875968992248062) < 1e-9, '')
kiem('Không có cảnh báo: precision là None', chi_so(10, 0, 5, 0)['precision'] is None, 'Khi TP + FP = 0, phép chia không xác định.')
kiem('F1 tính hai cách cho cùng kết quả', abs(c['f1'] - 2 * c['precision'] * c['recall'] / (c['precision'] + c['recall'])) < 1e-12, '')

## Bước 4 · Quét ngưỡng trên validation (Chặng 3)

Lập bảng `bang` cho m = 1 … 12 với các cột `m, tn, fp, fn, tp, canh_bao, precision, recall, f1`. Rồi vẽ hai hình: đường FP–FN và precision, recall theo m.

In [ ]:
# TODO: bảng 12 dòng như mô tả
bang = ...
raise NotImplementedError('TODO bước 4: lập bảng rồi xoá dòng raise này')
bang

In [ ]:
# TỰ KIỂM · bước 4
kiem('Bảng có 12 ngưỡng', len(bang) == 12 and list(bang.m) == list(range(1, 13)), '')
kiem('m = 4, 6, 8 khớp slide', [tuple(bang.loc[bang.m == k, ['fp', 'fn']].iloc[0]) for k in (4, 6, 8)] == [(18, 12), (9, 16), (4, 19)], '')
kiem('Số cảnh báo và recall không tăng khi nâng m', bang.canh_bao.is_monotonic_decreasing and bang.recall.is_monotonic_decreasing, 'Nâng m chỉ làm tin rời vùng gắn cờ.')
kiem('Có ít nhất một chỗ precision giảm khi nâng m', any(a > b for a, b in zip(bang.precision[:-1], bang.precision[1:]) if a is not None and b is not None and not pd.isna(b)),
     'Xem m = 8 → 9.')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(bang.fp, bang.fn, '-o')
for r in bang.itertuples():
    ax[0].annotate(f'm={r.m}', (r.fp, r.fn), textcoords='offset points', xytext=(4, 4), fontsize=8)
ax[0].set_xlim(0, 40); ax[0].set_ylim(0, 55); ax[0].set_xlabel('FP (validation)'); ax[0].set_ylabel('FN (validation)')
ax[1].plot(bang.m[:-1], bang.precision[:-1], '-o', label='precision'); ax[1].plot(bang.m, bang.recall, '-s', label='recall')
ax[1].set_xlabel('ngưỡng m'); ax[1].legend(); plt.tight_layout(); plt.show()

## Bước 5 · Chọn ngưỡng theo chi phí ghi trước, rồi khoá (Chặng 4)

Viết `chon_nguong(bang, c_fp, c_fn)` trả về m có chi phí `c_fp·FP + c_fn·FN` nhỏ nhất. **Quy tắc hoà ghi trước:** chi phí bằng nhau thì chọn ít cảnh báo hơn, rồi m lớn hơn. Mức cơ bản dùng `KICH_BAN = (5, 1)`. Mức khá khoá thêm ngưỡng cho `KICH_BAN_KHA = (1, 5)` **trước** khi chấm test.

In [ ]:
KICH_BAN = (5, 1)       # (c_FP, c_FN): ghi TRƯỚC khi nhìn kết quả
KICH_BAN_KHA = (1, 5)   # mức khá: kịch bản thứ hai, cũng ghi trước
LAM_MUC_KHA = True      # mức cơ bản: đặt False

def chon_nguong(bang, c_fp, c_fn):
    # TODO: trả về (m, chi_phi) theo quy tắc hoà ở trên
    raise NotImplementedError('TODO bước 5: viết chon_nguong')

NGUONG_KHOA, CHI_PHI_VAL = chon_nguong(bang, *KICH_BAN)
print('Ngưỡng đã khoá:', NGUONG_KHOA, '· chi phí trên validation:', CHI_PHI_VAL)
if LAM_MUC_KHA:
    NGUONG_KHA, CHI_PHI_KHA = chon_nguong(bang, *KICH_BAN_KHA)
    print('Ngưỡng mức khá đã khoá:', NGUONG_KHA, '· chi phí trên validation:', CHI_PHI_KHA)

In [ ]:
# TỰ KIỂM · bước 5
kiem('5·FP + FN chọn m = 8 (chi phí 39)', chon_nguong(bang, 5, 1) == (8, 39), 'Kiểm công thức chi phí và quy tắc hoà.')
kiem('FP + 5·FN chọn m = 4 (chi phí 78)', chon_nguong(bang, 1, 5) == (4, 78), '')
kiem('Nhân hai hệ số với 10 không đổi ngưỡng', chon_nguong(bang, 50, 10)[0] == chon_nguong(bang, 5, 1)[0], 'Chỉ tỷ lệ c_FP : c_FN quyết định.')

## Bước 6 · Chấm test một lần, so mốc, đọc tin sai (Chặng 4)

`cham_test` chỉ cho chấm **một lần**; mức khá truyền cả hai ngưỡng đã khoá vào **cùng một lần gọi**. Để so với ngưỡng đa số mà không chấm test lần hai, dùng kết quả Bài 2 đã báo: m = 6 cho TN · FP · FN · TP = 891 · 13 · 17 · 111. Mốc “luôn ham” chỉ cần nhãn, không cần dự đoán.

Lưu ý trung thực: test này đã được chấm ở Bài 2 (test lịch sử). Báo cáo con số như một minh hoạ việc áp quy tắc đã khoá, không phải đánh giá độc lập mới.

In [ ]:
_SO_LAN_CHAM = 0
BAI2_M6 = (891, 13, 17, 111)   # Bài 2 đã báo, không chấm lại

def cham_test(*cac_nguong):
    # Chấm test một lần cho các ngưỡng ĐÃ KHOÁ trên validation; trả dict m -> (tn, fp, fn, tp)
    global _SO_LAN_CHAM
    if _SO_LAN_CHAM >= 1:
        raise RuntimeError('Test đã chấm một lần. Muốn thử thêm: quay lại validation và ghi rõ trong báo cáo.')
    _SO_LAN_CHAM += 1
    return {m: ma_tran(test.y, test.votes >= m) for m in cac_nguong}

KQ = cham_test(NGUONG_KHOA, *([NGUONG_KHA] if LAM_MUC_KHA else []))
ket_qua = KQ[NGUONG_KHOA]
for m, mt in KQ.items():
    print('Test · m =', m, '· TN, FP, FN, TP =', mt, '·', {k: round(v, 4) for k, v in chi_so(*mt).items()})
print('So với m = 6 (Bài 2):', BAI2_M6, '· mốc luôn ham: accuracy', round((test.y == 0).mean(), 4), ', bắt được 0 spam')

In [ ]:
# TỰ KIỂM · bước 6
kiem('cham_test được gọi đúng một lần', _SO_LAN_CHAM == 1, 'Chạy lại toàn bộ notebook (Restart & Run All) nếu đã gọi nhiều lần.')
if KICH_BAN == (5, 1):
    kiem('Test với m = 8: (897, 7, 21, 107)', ket_qua == (897, 7, 21, 107), '')
if LAM_MUC_KHA and KICH_BAN_KHA == (1, 5):
    kiem('Mức khá · test với m = 4: (881, 23, 14, 114)', KQ[NGUONG_KHA] == (881, 23, 14, 114), 'Khoá cả hai ngưỡng trên validation rồi gọi cham_test một lần.')

In [ ]:
# Đọc ba tin sai của ngưỡng đã khoá (chỉ để mô tả, không suy diễn nhân quả)
test['du_doan'] = (test.votes >= NGUONG_KHOA).astype(int)
sai = test[test.y != test.du_doan].sort_values('source_record')
for r in pd.concat([sai[sai.y == 0].head(2), sai[sai.y == 1].head(1)]).itertuples():
    print(f'[{"FP" if r.y == 0 else "FN"}] bản ghi {r.source_record} · {r.votes}/11 phiếu · {r.text[:140]}')

### Đối chiếu với scikit-learn (mức khá)

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
khop = 0
for m in range(1, 13):
    p = (val.votes >= m).astype(int)
    tn, fp, fn, tp = confusion_matrix(val.y, p, labels=[0, 1]).ravel()
    c = chi_so(tn, fp, fn, tp)
    ok = (tn, fp, fn, tp) == ma_tran(val.y, p) and abs((c['recall'] or 0) - recall_score(val.y, p, zero_division=0)) < 1e-12 \
        and abs((c['precision'] or 0) - precision_score(val.y, p, zero_division=0)) < 1e-12 and abs((c['f1'] or 0) - f1_score(val.y, p, zero_division=0)) < 1e-12
    khop += ok
kiem('Hàm tự viết khớp scikit-learn ở cả 12 ngưỡng', khop == 12, 'Với m = 12, scikit-learn trả 0 khi zero_division=0; hàm của bạn trả None.')

## Báo cáo một trang (nộp cùng notebook)

Mỗi mục 2–4 câu.

1. **Dữ liệu và quy ước.** Tập nào, bao nhiêu tin, bao nhiêu spam; lớp dương; thứ tự nhãn.
2. **Mục tiêu và tiêu chí.** Kịch bản chi phí (ghi rõ là giả định), quy tắc hoà, tập dùng để chọn.
3. **Kết quả.** Bảng validation (ít nhất m = 4, 6, 8 và ngưỡng đã chọn); ngưỡng đã khoá; bốn ô trên test, P, R, F1, số cảnh báo; so với m = 6 và mốc “luôn ham”.
4. **Đánh đổi.** So với m = 6, đổi bao nhiêu FP lấy bao nhiêu FN.
5. **Ba tin sai.** Mô tả vì sao hai đặc trưng không phân biệt được (không suy diễn nhân quả).
6. **Giới hạn.** Ít nhất ba: chi phí giả định; test lịch sử; validation đã dùng hai lần; corpus tiếng Anh 2011, chia ngẫu nhiên; hai đặc trưng hình thức.

## Nâng cao (tuỳ chọn)

- **AUC theo định nghĩa so cặp** trên validation: tỷ lệ cặp (spam, ham) mà spam có số phiếu cao hơn, cộng nửa điểm cho cặp hoà; so với `roc_auc_score`.
- **AP theo bậc recall:** AP = Σ (Rₙ − Rₙ₋₁)·Pₙ qua các ngưỡng; so với `average_precision_score`.
- **Tỷ lệ lớp:** giữ TPR, FPR của ngưỡng đã chọn trên validation; tính precision nếu tỷ lệ spam chỉ là 2%: P = π·TPR / (π·TPR + (1 − π)·FPR).